In [1]:
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

In [2]:
input_file = "eng_updated selected dataset.xlsx"

df = pd.read_excel(input_file)
df.head()

,sentence_id,orginal_sen
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [3]:
english_col = "orginal_sen"

if "sentence_id" not in df.columns:
    df["sentence_id"] = range(1, len(df) + 1)

df = df[["sentence_id", english_col]].copy()
df.rename(columns={english_col: "en_original"}, inplace=True)

df.head()

,sentence_id,en_original
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [4]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("NLLB model loaded")

NLLB model loaded


In [5]:
src_lang = "eng_Latn"
tgt_lang = "hin_Deva"

In [7]:
def translate_nllb(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang)
    )
    
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [8]:
tqdm.pandas()

df["hi_translation"] = df["en_original"].progress_apply(
    lambda x: translate_nllb(x, "eng_Latn", "hin_Deva")
)

df.head()

100%|██████████| 200/200 [06:45<00:00,  2.03s/it]


,sentence_id,en_original,hi_translation
0,1,Let's try something.,चलो कुछ कोशिश करते हैं.
1,2,I have to go to sleep.,मुझे सोने जाना है.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और यह म्यूरियल का जन्मदिन है!
3,4,Muiriel is 20 now.,मुरियल अब 20 साल का है।
4,5,"The password is ""Muiriel"".","पासवर्ड ""Muiriel"" है।"


In [9]:
df["en_backtranslation"] = df["hi_translation"].progress_apply(
    lambda x: translate_nllb(x, "hin_Deva", "eng_Latn")
)

df.head()

100%|██████████| 200/200 [06:15<00:00,  1.88s/it]


,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करते हैं.,Let's try something.
1,2,I have to go to sleep.,मुझे सोने जाना है.,I have to go to bed.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और यह म्यूरियल का जन्मदिन है!,Today is June 18th and it's Muriel's birthday!
3,4,Muiriel is 20 now.,मुरियल अब 20 साल का है।,Muriel is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड ""Muiriel"" है।","The password is ""Muiriel""."


In [10]:
df.to_excel("nllb_bt.xlsx", index=False)
print("Saved nllb_bt.xlsx")

Saved nllb_bt.xlsx
